# Teapot Image Preprocessing

这个 notebook 分成四部分：

1. 在 target 坐标正下方 `200` 像素处画一个 `50x50` 的 background box，用它估计 background noise。
2. 以 `F475W` 的 lens 中心为参考裁切两个通道的 lens cutout；`F814W` 的中心通过 `F475W -> RA/Dec -> F814W` 映射后四舍五入到整数像素。
3. 自动标出两个通道的 point-source candidates，显示 `psf_id` 供挑选。
4. 根据选定的 `psf_id`，用“单颗基准星 + 全局 correction field”的 SVI 方法生成 PSF，并保存成 FITS。

输出文件直接写在同一个 `RUN_TAG` 目录里，不再分 `mask_inputs / point_sources / psf` 子目录。


In [ ]:
import os
import sys
import json
from pathlib import Path

os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Rectangle
from IPython.display import display
from astropy.io import fits
from astropy.table import Table
from astropy.wcs import WCS
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder

import jax
import jax.numpy as jnp
from jax.scipy.ndimage import map_coordinates

import numpyro
import numpyro.distributions as dist
import numpyro.infer as infer
import numpyro.infer.autoguide as autoguide
import optax

plt.rcParams["figure.dpi"] = 120
np.random.seed(0)
jax.config.update("jax_enable_x64", True)
numpyro.enable_x64()

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "Tian_infra.py").exists():
    for candidate in [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents):
        if (candidate / "Tian_infra.py").exists():
            NOTEBOOK_DIR = candidate
            break
if not (NOTEBOOK_DIR / "Tian_infra.py").exists():
    raise FileNotFoundError("Cannot locate Tian_infra.py from the current working directory.")
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

PROJECT_ROOT = NOTEBOOK_DIR.parents[2]
DATA_DIR = PROJECT_ROOT / "Herculens" / "Data" / "Teapot"
RUN_TAG = "prep_20260419_bgbox_psfsvi_v2_flat"
PREPROCESS_ROOT = NOTEBOOK_DIR / "preprocessed"
OUTPUT_DIR = PREPROCESS_ROOT / RUN_TAG

for path in [OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

from power_spectrum_prior import P_Matern, pack_fft_values, K_grid

BAND_FILES = {
    "f475w": DATA_DIR / "ifo601010_drc.fits",
    "f814w": DATA_DIR / "ifo602010_drc.fits",
}

F475W_CENTER_PIX = (2062.8, 1151.0)
SCIENCE_CUTOUT_SIZE = (100, 100)
PSF_STAMP_SIZE = 49
PSF_SUPERSAMPLING = 1
DETECTION_FWHM_PIX = 3.0
DETECTION_THRESHOLD_SIGMA = 20.0
DETECTION_BORDER_MARGIN = PSF_STAMP_SIZE // 2 + 1
BG_OFFSET_PIX = 200
BG_BOX_SIZE = 50
BG_MIN_VALID_FRACTION = 0.9

SELECTED_PSF_IDS = {
    "f475w": [0, 1, 2, 5, 6, 7, 8, 9],
    "f814w": [0, 1, 2, 5, 6, 7, 8, 9],
}

DEFAULT_PSF_REFERENCE_PIX = {
    "f475w": (3463.9, 1583.9),
    "f814w": (3470.5, 1558.6),
}

BASE_PSF_POLICY = "brightest_selected_star"
PSF_SVI_SEED = 123
PSF_SVI_MAX_ITER = 3000
PSF_SVI_LR_INIT = 0.01
PSF_SVI_TRANSITION_STEPS = (200, 20)
PSF_GUIDE_INIT_SCALE = 0.03
PSF_XY_PRIOR_SIGMA = 0.35
PSF_XY_BOUNDS = (-1.5, 1.5)
PSF_POS_FLOOR = 1e-12
PSF_MULT_LOG_CLIP = 8.0
PSF_POWER_N_MAX = 80.0


def make_log_norm(data, lo=5.0, hi=99.8, floor=1e-6):
    finite = np.asarray(data, dtype=float)
    finite = finite[np.isfinite(finite) & (finite > 0)]
    if finite.size == 0:
        return LogNorm(vmin=floor, vmax=1.0)
    vmin = max(np.percentile(finite, lo), floor)
    vmax = max(np.percentile(finite, hi), vmin * 1.05)
    return LogNorm(vmin=vmin, vmax=vmax)

def estimate_background_from_box(data, target_center_xy, offset_pix, box_size, min_valid_fraction=0.9):
    background_center_xy = (
        int(np.rint(target_center_xy[0])),
        int(np.rint(target_center_xy[1] - offset_pix)),
    )
    x0, x1, y0, y1, cx, cy = integer_bounds(background_center_xy, (box_size, box_size))
    if x0 < 0 or y0 < 0 or x1 > data.shape[1] or y1 > data.shape[0]:
        raise ValueError(
            f"Background box around {(cx, cy)} with size {(box_size, box_size)} falls outside image shape {data.shape}."
        )
    background_box = data[y0:y1, x0:x1].copy()
    valid = np.isfinite(background_box)
    valid_fraction = float(np.mean(valid))
    if valid_fraction < min_valid_fraction:
        raise ValueError("Background box does not contain enough valid pixels.")
    box_mean, box_median, box_std = sigma_clipped_stats(background_box, sigma=3.0, maxiters=10)
    box_info = {
        "center_x": int(cx),
        "center_y": int(cy),
        "x0": int(x0),
        "x1": int(x1),
        "y0": int(y0),
        "y1": int(y1),
        "mean": float(box_mean),
        "median": float(box_median),
        "std": float(box_std),
        "valid_fraction": valid_fraction,
    }
    return background_box, box_info


def integer_bounds(center_xy, size_xy):
    cx = int(np.rint(center_xy[0]))
    cy = int(np.rint(center_xy[1]))
    sx = int(size_xy[0])
    sy = int(size_xy[1])
    hx = sx // 2
    hy = sy // 2
    x0 = cx - hx
    y0 = cy - hy
    x1 = x0 + sx
    y1 = y0 + sy
    return x0, x1, y0, y1, cx, cy


def extract_integer_cutout(data, header, center_xy, size_xy):
    x0, x1, y0, y1, cx, cy = integer_bounds(center_xy, size_xy)
    if x0 < 0 or y0 < 0 or x1 > data.shape[1] or y1 > data.shape[0]:
        raise ValueError(
            f"Cutout around {(cx, cy)} with size {size_xy} falls outside image shape {data.shape}."
        )
    cutout = data[y0:y1, x0:x1].copy()
    sliced_wcs = WCS(header).slice((slice(y0, y1), slice(x0, x1)))
    cutout_header = header.copy()
    cutout_header.update(sliced_wcs.to_header())
    cutout_header["CUTX0"] = x0
    cutout_header["CUTX1"] = x1
    cutout_header["CUTY0"] = y0
    cutout_header["CUTY1"] = y1
    cutout_header["CENXPIX"] = cx
    cutout_header["CENYPIX"] = cy
    return cutout, cutout_header, {
        "x0": x0,
        "x1": x1,
        "y0": y0,
        "y1": y1,
        "center_x": cx,
        "center_y": cy,
    }


def save_fits(data, header, path, extname):
    hdu = fits.PrimaryHDU(data=np.asarray(data, dtype=np.float32), header=header.copy())
    hdu.header["EXTNAME"] = extname
    hdu.writeto(path, overwrite=True)


def save_preview_png(data, output_path, title):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(data, origin="lower", cmap="twilight", norm=make_log_norm(data))
    ax.set_title(title)
    ax.set_xlabel("x [pix]")
    ax.set_ylabel("y [pix]")
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close(fig)


def empty_candidate_table():
    return Table(
        names=["psf_id", "xcentroid", "ycentroid", "flux", "peak", "sharpness", "roundness1", "roundness2"],
        dtype=[int, float, float, float, float, float, float, float],
    )


def detect_point_sources(data, threshold_sigma, fwhm_pix, border_margin, background_std=None):
    _, median, clipped_std = sigma_clipped_stats(data, sigma=3.0, maxiters=10)
    std = float(background_std) if background_std is not None else float(clipped_std)
    finder = DAOStarFinder(
        fwhm=fwhm_pix,
        threshold=threshold_sigma * std,
        exclude_border=True,
    )
    detected = finder(data - median)
    if detected is None:
        return empty_candidate_table(), std
    x = np.asarray(detected["xcentroid"], dtype=float)
    y = np.asarray(detected["ycentroid"], dtype=float)
    flux = np.asarray(detected["flux"], dtype=float)
    sharp = np.asarray(detected["sharpness"], dtype=float)
    round1 = np.asarray(detected["roundness1"], dtype=float)
    round2 = np.asarray(detected["roundness2"], dtype=float)
    keep = (
        np.isfinite(x)
        & np.isfinite(y)
        & np.isfinite(flux)
        & (flux > 0)
        & (sharp > 0.4)
        & (sharp < 1.0)
        & (np.abs(round1) < 0.8)
        & (np.abs(round2) < 0.8)
        & (x > border_margin)
        & (x < data.shape[1] - border_margin)
        & (y > border_margin)
        & (y < data.shape[0] - border_margin)
    )
    filtered = detected[keep]
    if len(filtered) == 0:
        return empty_candidate_table(), std
    order = np.argsort(np.asarray(filtered["flux"], dtype=float))[::-1]
    filtered = filtered[order]
    filtered["psf_id"] = np.arange(len(filtered), dtype=int)
    filtered = filtered[["psf_id", "xcentroid", "ycentroid", "flux", "peak", "sharpness", "roundness1", "roundness2"]]
    return filtered, std


def stamp_from_center(data, center_xy, size):
    x0, x1, y0, y1, cx, cy = integer_bounds(center_xy, (size, size))
    if x0 < 0 or y0 < 0 or x1 > data.shape[1] or y1 > data.shape[0]:
        raise ValueError(
            f"Stamp around {(cx, cy)} with size {size} falls outside image shape {data.shape}."
        )
    return data[y0:y1, x0:x1].copy()


def display_candidate_pages(band, data, table, stamp_size, per_page=30, ncols=5):
    if len(table) == 0:
        print(f"No candidates detected for {band}.")
        return
    num_pages = int(np.ceil(len(table) / per_page))
    for page in range(num_pages):
        start = page * per_page
        stop = min((page + 1) * per_page, len(table))
        subset = table[start:stop]
        nrows = int(np.ceil(len(subset) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 3.0 * nrows))
        axes = np.atleast_1d(axes).ravel()
        for ax in axes:
            ax.axis("off")
        for ax, row in zip(axes, subset):
            stamp = stamp_from_center(
                data,
                (float(row["xcentroid"]), float(row["ycentroid"])),
                stamp_size,
            )
            ax.imshow(stamp, origin="lower", cmap="twilight", norm=make_log_norm(stamp))
            ax.set_title(f'{band.upper()} ID {int(row["psf_id"])}', fontsize=9)
            ax.set_xticks([])
            ax.set_yticks([])
        fig.suptitle(f"{band.upper()} point-source candidates | page {page + 1}/{num_pages}")
        fig.tight_layout()
        plt.show()


def resolve_selected_psf_ids(band, candidate_table, explicit_ids, default_reference_pix):
    if explicit_ids:
        return [int(pid) for pid in explicit_ids], "manual"
    if default_reference_pix is None:
        raise ValueError(f"No PSF IDs configured for {band}.")
    if len(candidate_table) == 0:
        raise ValueError(f"No point-source candidates found for {band}.")
    x = np.asarray(candidate_table["xcentroid"], dtype=float)
    y = np.asarray(candidate_table["ycentroid"], dtype=float)
    dist_pix = np.hypot(x - default_reference_pix[0], y - default_reference_pix[1])
    best = int(np.argmin(dist_pix))
    return [int(candidate_table[best]["psf_id"])], "nearest_to_default_reference"


def split_scheduler(max_iterations, init_value=0.01, decay_rates=(0.99, 0.98), transition_steps=(200, 20), boundary=0.8):
    boundary = int(max_iterations * boundary)
    scheduler1 = optax.exponential_decay(
        init_value=init_value,
        decay_rate=decay_rates[0],
        transition_steps=transition_steps[0],
    )
    scheduler2 = optax.exponential_decay(
        init_value=scheduler1(boundary),
        decay_rate=decay_rates[1],
        transition_steps=transition_steps[1],
    )
    return optax.join_schedules([scheduler1, scheduler2], boundaries=[boundary])


def normalize_kernel_np(kernel, eps=1e-20):
    arr = np.asarray(kernel, dtype=float)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = np.clip(arr, 0.0, None)
    total = float(np.sum(arr))
    if total <= eps:
        raise ValueError("Kernel normalization failed because the total flux is non-positive.")
    return arr / total


def normalize_kernel_jax(kernel, eps=1e-20):
    kernel = jnp.nan_to_num(kernel, nan=0.0, posinf=0.0, neginf=0.0)
    kernel = jnp.clip(kernel, 0.0, None)
    total = jnp.sum(kernel)
    return jnp.where(total > eps, kernel / total, kernel)


def build_poisson_plus_background_noise(science_stamp, background_sigma_scalar, background_level_scalar, exptime_sec):
    science_stamp = np.asarray(science_stamp, dtype=float)
    source_rate = np.clip(science_stamp - float(background_level_scalar), 0.0, None)
    poisson_sigma = np.sqrt(source_rate / max(float(exptime_sec), 1e-12))
    total_sigma = np.sqrt(float(background_sigma_scalar) ** 2 + poisson_sigma ** 2)
    return np.clip(total_sigma, 1e-8, None)


def choose_base_star_id(candidate_table, selected_ids, policy="brightest_selected_star"):
    selected_ids = [int(pid) for pid in selected_ids]
    subset = candidate_table[np.isin(np.asarray(candidate_table["psf_id"], dtype=int), selected_ids)]
    if len(subset) != len(selected_ids):
        missing = sorted(set(selected_ids) - set(np.asarray(subset["psf_id"], dtype=int).tolist()))
        raise ValueError(f"Missing selected psf_id values in candidate table: {missing}")
    if policy == "first_selected_star":
        return int(selected_ids[0])
    if policy == "brightest_selected_star":
        best = int(np.argmax(np.asarray(subset["flux"], dtype=float)))
        return int(subset[best]["psf_id"])
    raise ValueError(f"Unsupported BASE_PSF_POLICY: {policy}")


def build_initial_base_psf(science, candidate_table, selected_ids, background_level_scalar, stamp_size, policy):
    base_pid = choose_base_star_id(candidate_table, selected_ids, policy=policy)
    match = candidate_table[candidate_table["psf_id"] == base_pid]
    if len(match) != 1:
        raise ValueError(f"Could not uniquely locate base psf_id={base_pid}.")
    row = match[0]
    center = (float(row["xcentroid"]), float(row["ycentroid"]))
    base_stamp_raw = stamp_from_center(science, center, stamp_size)
    base_stamp_bgsub = np.clip(base_stamp_raw - float(background_level_scalar), 0.0, None)
    base_psf_det = normalize_kernel_np(base_stamp_bgsub + PSF_POS_FLOOR)
    return base_pid, center, base_stamp_raw, base_psf_det


def weighted_ls_flux(unit_model_stack, data_minus_bkg, err_stack, mask_data, eps=1e-12):
    inv_var = mask_data / (err_stack**2 + eps)
    numer = jnp.sum(unit_model_stack * data_minus_bkg * inv_var, axis=(1, 2))
    denom = jnp.sum((unit_model_stack**2) * inv_var, axis=(1, 2)) + eps
    flux = numer / denom
    return jnp.clip(flux, eps, None)


def project_zero_moments(corr, eps=1e-12):
    ny, nx = corr.shape
    yy, xx = jnp.indices((ny, nx), dtype=corr.dtype)
    xx = xx - (nx - 1) / 2.0
    yy = yy - (ny - 1) / 2.0
    design = jnp.stack([
        jnp.ones_like(corr).reshape(-1),
        xx.reshape(-1),
        yy.reshape(-1),
    ], axis=1)
    data = corr.reshape(-1)
    lhs = design.T @ design + eps * jnp.eye(3, dtype=corr.dtype)
    rhs = design.T @ data
    coeff = jnp.linalg.solve(lhs, rhs)
    projected = data - design @ coeff
    return projected.reshape(ny, nx)


def matern_correction_field(param_name, k_values, n_high=80.0):
    with numpyro.plate(f"{param_name}_hyper", 1):
        n = numpyro.sample(f"n_{param_name}", dist.LogUniform(0.3, n_high))
        sigma = numpyro.sample(f"sigma_{param_name}", dist.LogUniform(1e-5, 3.0))
        rho = numpyro.sample(f"rho_{param_name}", dist.LogNormal(jnp.log(3.0), 0.8))
    power = P_Matern(k_values, n[0], sigma[0], rho[0], k_zero=0.0)
    scale = jnp.sqrt(power)
    ny, nx = scale.shape
    with numpyro.plate(f"{param_name}_fft_y", ny):
        with numpyro.plate(f"{param_name}_fft_x", nx):
            pixels_wn = numpyro.sample(f"pixels_wn_{param_name}", dist.Normal(0, 1))
    correction = jnp.fft.irfft2(pack_fft_values(pixels_wn * scale), s=scale.shape, norm="ortho")
    return project_zero_moments(correction)


def render_shifted_kernel(kernel_det, shift_x_pix, shift_y_pix):
    y_idx, x_idx = jnp.indices(kernel_det.shape, dtype=kernel_det.dtype)
    sample_y = y_idx - shift_y_pix
    sample_x = x_idx - shift_x_pix
    return map_coordinates(kernel_det, [sample_y, sample_x], order=1, mode="nearest")


## 1. 估计 background noise

这里不生成任何 noise map。background noise 只用一个 box 来估计：以 target 中心为基准，沿着图像正下方偏移 `200` 像素，在那里放一个 `50x50` 的正方形，对这个 box 做 sigma-clipped 统计，并记录它的 `mean / median / std`。

In [ ]:
band_data = {}

for band, path in BAND_FILES.items():
    with fits.open(path, memmap=True) as hdul:
        primary_header = hdul[0].header.copy()
        science = hdul[1].data.astype(float)
        science_header = hdul[1].header.copy()
    exptime_sec = float(primary_header.get("EXPTIME", primary_header.get("TEXPTIME", 1.0)))
    band_data[band] = {
        "science": science,
        "science_header": science_header,
        "primary_header": primary_header,
        "exptime_sec": exptime_sec,
        "path": path,
    }

wcs_f475w = WCS(band_data["f475w"]["science_header"])
lens_ra_deg, lens_dec_deg = wcs_f475w.pixel_to_world_values(*F475W_CENTER_PIX)
wcs_f814w = WCS(band_data["f814w"]["science_header"])
f814w_center_float = wcs_f814w.world_to_pixel_values(float(lens_ra_deg), float(lens_dec_deg))

target_centers = {
    "f475w": (int(np.rint(F475W_CENTER_PIX[0])), int(np.rint(F475W_CENTER_PIX[1]))),
    "f814w": (int(np.rint(f814w_center_float[0])), int(np.rint(f814w_center_float[1]))),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for row_idx, band in enumerate(["f475w", "f814w"]):
    science = band_data[band]["science"]
    science_header = band_data[band]["science_header"]
    background_box, background_info = estimate_background_from_box(
        science,
        target_center_xy=target_centers[band],
        offset_pix=BG_OFFSET_PIX,
        box_size=BG_BOX_SIZE,
        min_valid_fraction=BG_MIN_VALID_FRACTION,
    )
    background_box_header = science_header.copy()
    background_box_header["BGOFFSET"] = (BG_OFFSET_PIX, "Background-box offset below target in pixels")
    background_box_header["BGSIZE"] = (BG_BOX_SIZE, "Background-box size in pixels")
    background_box_header["BGMED"] = (background_info["median"], "Sigma-clipped background median")
    background_box_header["BGSTD"] = (background_info["std"], "Sigma-clipped background std")
    background_box_header["EXPTIME"] = (band_data[band]["exptime_sec"], "Exposure time from primary header")
    background_box_path = OUTPUT_DIR / f"{band}_background_box_{BG_BOX_SIZE}x{BG_BOX_SIZE}.fits"
    save_fits(background_box, background_box_header, background_box_path, "BGCUTOUT")
    band_data[band]["target_center_pix"] = target_centers[band]
    band_data[band]["background_box"] = background_box
    band_data[band]["background_box_header"] = background_box_header
    band_data[band]["background_box_path"] = background_box_path
    band_data[band]["background_info"] = background_info
    band_data[band]["background_sigma_scalar"] = background_info["std"]
    band_data[band]["background_level_scalar"] = background_info["median"]

    preview_size = 500
    px0, px1, py0, py1, _, _ = integer_bounds(target_centers[band], (preview_size, preview_size))
    preview = science[py0:py1, px0:px1]
    axes[row_idx, 0].imshow(preview, origin="lower", cmap="twilight", norm=make_log_norm(preview))
    axes[row_idx, 0].scatter(target_centers[band][0] - px0, target_centers[band][1] - py0, s=30, c="red", marker="x")
    axes[row_idx, 0].add_patch(
        Rectangle(
            (background_info["x0"] - px0, background_info["y0"] - py0),
            BG_BOX_SIZE,
            BG_BOX_SIZE,
            fill=False,
            edgecolor="cyan",
            linewidth=1.2,
            linestyle="--",
        )
    )
    axes[row_idx, 0].set_title(f"{band.upper()} target and background box")
    axes[row_idx, 0].set_xlabel("preview x [pix]")
    axes[row_idx, 0].set_ylabel("preview y [pix]")

    axes[row_idx, 1].imshow(background_box, origin="lower", cmap="twilight", norm=make_log_norm(background_box))
    axes[row_idx, 1].set_title(
        f"{band.upper()} background box | median={background_info['median']:.4g}, std={background_info['std']:.4g}"
    )
    axes[row_idx, 1].set_xlabel("box x [pix]")
    axes[row_idx, 1].set_ylabel("box y [pix]")

    print(
        f"{band}: target center = {target_centers[band]} | background box center = "
        f"({background_info['center_x']}, {background_info['center_y']}) | "
        f"median = {background_info['median']:.6g} e-/s | std = {background_info['std']:.6g} e-/s | "
        f"exptime = {band_data[band]['exptime_sec']:.1f} s | box -> {background_box_path}"
    )

fig.tight_layout()
plt.show()


## 2. 裁切 lens cutout

以 `F475W` 的 lens 中心 `(2062.8, 1151.0)` 为参考，先转成 sky 坐标，再映射到 `F814W`，最后直接取整裁切；整个过程不做重投影，也不做插值。

In [ ]:
wcs_f475w = WCS(band_data["f475w"]["science_header"])
lens_ra_deg, lens_dec_deg = wcs_f475w.pixel_to_world_values(*F475W_CENTER_PIX)

wcs_f814w = WCS(band_data["f814w"]["science_header"])
f814w_center_float = wcs_f814w.world_to_pixel_values(float(lens_ra_deg), float(lens_dec_deg))

cutout_centers = {
    "f475w": band_data["f475w"]["target_center_pix"],
    "f814w": band_data["f814w"]["target_center_pix"],
}

metadata = {
    "run_tag": RUN_TAG,
    "f475w_center_input_pix": [float(F475W_CENTER_PIX[0]), float(F475W_CENTER_PIX[1])],
    "lens_radec_deg": [float(lens_ra_deg), float(lens_dec_deg)],
    "f814w_center_from_radec_float": [float(f814w_center_float[0]), float(f814w_center_float[1])],
    "science_cutout_size": list(SCIENCE_CUTOUT_SIZE),
    "background_estimation": {
        "offset_below_target_pix": BG_OFFSET_PIX,
        "box_size_pix": BG_BOX_SIZE,
        "min_valid_fraction": BG_MIN_VALID_FRACTION,
    },
    "cutouts": {},
}

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

for ax, band in zip(axes, ["f475w", "f814w"]):
    science_cutout, science_cutout_header, cutout_meta = extract_integer_cutout(
        band_data[band]["science"],
        band_data[band]["science_header"],
        cutout_centers[band],
        SCIENCE_CUTOUT_SIZE,
    )
    science_path = OUTPUT_DIR / f"{band}_lens_cutout_{SCIENCE_CUTOUT_SIZE[0]}x{SCIENCE_CUTOUT_SIZE[1]}.fits"
    save_fits(science_cutout, science_cutout_header, science_path, "SCI")
    band_data[band]["science_cutout"] = science_cutout
    band_data[band]["science_cutout_path"] = science_path
    metadata["cutouts"][band] = {
        "center_pix": [int(cutout_meta["center_x"]), int(cutout_meta["center_y"])],
        "bounds": cutout_meta,
        "science_path": str(science_path),
        "background_box_path": str(band_data[band]["background_box_path"]),
        "background_box_info": band_data[band]["background_info"],
        "background_sigma_scalar": float(band_data[band]["background_sigma_scalar"]),
        "background_level_scalar": float(band_data[band]["background_level_scalar"]),
        "exptime_sec": float(band_data[band]["exptime_sec"]),
    }
    ax.imshow(science_cutout, origin="lower", cmap="twilight", norm=make_log_norm(science_cutout))
    ax.set_title(f"{band.upper()} lens cutout")
    ax.set_xlabel("x [pix]")
    ax.set_ylabel("y [pix]")

metadata_path = OUTPUT_DIR / "preprocessing_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as fh:
    json.dump(metadata, fh, indent=2)

fig.tight_layout()
plt.show()
print(json.dumps(metadata, indent=2))


## 3. 自动检测并显示 point-source candidates

这里会在整张 drc 图上自动找点源候选，并保存一个 `psf_id` 列。你后面只需要在下一格里填你想用的 `psf_id` 即可。为了避免把明显的边缘源和扩展源一并塞进来，这里做了比较保守的 shape / border 过滤。

In [ ]:
candidate_tables = {}

for band in ["f475w", "f814w"]:
    table, clipped_std = detect_point_sources(
        band_data[band]["science"],
        threshold_sigma=DETECTION_THRESHOLD_SIGMA,
        fwhm_pix=DETECTION_FWHM_PIX,
        border_margin=DETECTION_BORDER_MARGIN,
        background_std=band_data[band]["background_sigma_scalar"],
    )
    candidate_tables[band] = table
    print(f"{band}: {len(table)} candidates | background std used = {clipped_std:.6g}")
    display(table[: min(10, len(table))])

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(band_data[band]["science"], origin="lower", cmap="twilight", norm=make_log_norm(band_data[band]["science"]))
    if len(table) > 0:
        x = np.asarray(table["xcentroid"], dtype=float)
        y = np.asarray(table["ycentroid"], dtype=float)
        ax.scatter(x, y, s=40, facecolors="none", edgecolors="cyan", linewidths=0.8)
        for row in table:
            ax.text(
                float(row["xcentroid"]) + 8,
                float(row["ycentroid"]) + 8,
                str(int(row["psf_id"])),
                color="white",
                fontsize=7,
                ha="left",
                va="bottom",
            )
    ax.set_title(f"{band.upper()} point-source candidates")
    ax.set_xlabel("x [pix]")
    ax.set_ylabel("y [pix]")
    fig.tight_layout()
    plt.show()

    display_candidate_pages(
        band,
        band_data[band]["science"],
        table,
        stamp_size=PSF_STAMP_SIZE,
        per_page=30,
        ncols=5,
    )


## 4. 选 `psf_id` 并用单星 base + correction field 的 SVI 生成 PSF

这一部分不再调用 `starred.build_psf`。现在的流程跟 `Herculensedquasar/WFI2033/Step_3_build_psf_svi.ipynb` 一致：

- 先在你选中的星里挑一颗作为 `base PSF`
- 再在这颗 base 上施加一个全局的 multiplicative correction field
- 对所有选中的星联合做 SVI
- 每颗星都有自己的 `x/y` 子像素偏移、背景和解析解 flux

这里的噪声也改成手工构造的每像素 `sigma`：

- `sigma_bg` 来自前面 background box 的 sigma-clipped `std`
- `sigma_poisson` 用 `sqrt(max(science - background, 0) / EXPTIME)` 计算，单位保持在 `e-/s`
- 最终 `sigma_tot = sqrt(sigma_bg^2 + sigma_poisson^2)`

`SELECTED_PSF_IDS` 保持为你指定的：

- `F475W`: `[0, 1, 2, 5, 6, 7, 8, 9]`
- `F814W`: `[0, 1, 2, 5, 6, 7, 8, 9]`


In [ ]:
resolved_psf_ids = {}

for band in ["f475w", "f814w"]:
    resolved_ids, source_tag = resolve_selected_psf_ids(
        band,
        candidate_tables[band],
        SELECTED_PSF_IDS[band],
        DEFAULT_PSF_REFERENCE_PIX[band],
    )
    resolved_psf_ids[band] = resolved_ids
    print(f"{band}: using PSF IDs {resolved_ids} ({source_tag})")

psf_summaries = {}

for band in ["f475w", "f814w"]:
    table = candidate_tables[band]
    science = band_data[band]["science"]
    background_sigma_scalar = float(band_data[band]["background_sigma_scalar"])
    background_level_scalar = float(band_data[band]["background_level_scalar"])
    exptime_sec = float(band_data[band]["exptime_sec"])

    image_stamps = []
    noise_stamps = []
    mask_stamps = []

    for pid in resolved_psf_ids[band]:
        match = table[table["psf_id"] == pid]
        if len(match) != 1:
            raise ValueError(f"Could not uniquely locate psf_id={pid} for {band}.")
        row = match[0]
        center = (float(row["xcentroid"]), float(row["ycentroid"]))
        science_stamp = stamp_from_center(science, center, PSF_STAMP_SIZE)
        noise_stamp = build_poisson_plus_background_noise(
            science_stamp,
            background_sigma_scalar=background_sigma_scalar,
            background_level_scalar=background_level_scalar,
            exptime_sec=exptime_sec,
        )
        stamp_mask = np.isfinite(science_stamp) & np.isfinite(noise_stamp) & (noise_stamp > 0)
        image_stamps.append(np.nan_to_num(science_stamp, nan=background_level_scalar))
        noise_stamps.append(noise_stamp)
        mask_stamps.append(stamp_mask.astype(float))

    base_pid, base_center, base_stamp_raw, base_psf_det = build_initial_base_psf(
        science,
        table,
        resolved_psf_ids[band],
        background_level_scalar=background_level_scalar,
        stamp_size=PSF_STAMP_SIZE,
        policy=BASE_PSF_POLICY,
    )

    sci_stack = jnp.asarray(np.asarray(image_stamps, dtype=np.float64))
    err_stack = jnp.asarray(np.asarray(noise_stamps, dtype=np.float64))
    mask_stack = jnp.asarray(np.asarray(mask_stamps, dtype=np.float64))
    bkg_loc = jnp.full((len(image_stamps),), background_level_scalar, dtype=jnp.float64)
    bkg_scale = jnp.full((len(image_stamps),), max(background_sigma_scalar, 1e-6), dtype=jnp.float64)
    base_psf_det_jax = jnp.asarray(base_psf_det, dtype=jnp.float64)
    k_values_det = jnp.asarray(K_grid(base_psf_det.shape).k, dtype=jnp.float64)

    def model_psf_svi(sci_data, err_data, mask_data, bkg_loc, bkg_scale, base_psf_det, k_values):
        corr_field = matern_correction_field("psf_corr", k_values, n_high=PSF_POWER_N_MAX)
        corr_field = numpyro.deterministic("corr_field", corr_field)
        log_multiplier = jnp.clip(corr_field, -PSF_MULT_LOG_CLIP, PSF_MULT_LOG_CLIP)
        psf_det_model = normalize_kernel_jax(base_psf_det * jnp.exp(log_multiplier) + PSF_POS_FLOOR)
        numpyro.deterministic("psf_det_model", psf_det_model)

        n_star = sci_data.shape[0]
        with numpyro.plate("stars", n_star):
            x_shift = numpyro.sample(
                "x_shift",
                dist.TruncatedNormal(
                    loc=jnp.zeros(n_star),
                    scale=PSF_XY_PRIOR_SIGMA,
                    low=PSF_XY_BOUNDS[0],
                    high=PSF_XY_BOUNDS[1],
                ),
            )
            y_shift = numpyro.sample(
                "y_shift",
                dist.TruncatedNormal(
                    loc=jnp.zeros(n_star),
                    scale=PSF_XY_PRIOR_SIGMA,
                    low=PSF_XY_BOUNDS[0],
                    high=PSF_XY_BOUNDS[1],
                ),
            )
            background = numpyro.sample("background", dist.Normal(loc=bkg_loc, scale=bkg_scale))

        unit_model_stack = jax.vmap(render_shifted_kernel, in_axes=(None, 0, 0))(
            psf_det_model,
            x_shift,
            y_shift,
        )
        data_minus_bkg = sci_data - background[:, None, None]
        flux_opt = weighted_ls_flux(unit_model_stack, data_minus_bkg, err_data, mask_data)
        model_stack = unit_model_stack * flux_opt[:, None, None] + background[:, None, None]

        numpyro.deterministic("flux_opt", flux_opt)
        numpyro.deterministic("model_stack", model_stack)

        logp = dist.Normal(model_stack, err_data).log_prob(sci_data)
        logp = jnp.where(mask_data > 0.5, logp, 0.0)
        numpyro.factor("obs_masked", jnp.sum(logp))

    rng_key = jax.random.PRNGKey(PSF_SVI_SEED)
    guide = autoguide.AutoDiagonalNormal(
        model_psf_svi,
        init_loc_fn=infer.init_to_median(num_samples=15),
        init_scale=PSF_GUIDE_INIT_SCALE,
    )
    scheduler = split_scheduler(
        PSF_SVI_MAX_ITER,
        init_value=PSF_SVI_LR_INIT,
        transition_steps=PSF_SVI_TRANSITION_STEPS,
    )
    optim = optax.adabelief(learning_rate=scheduler)
    svi = infer.SVI(model_psf_svi, guide, optim, infer.TraceMeanField_ELBO())
    svi_result = svi.run(
        rng_key,
        PSF_SVI_MAX_ITER,
        sci_stack,
        err_stack,
        mask_stack,
        bkg_loc,
        bkg_scale,
        base_psf_det_jax,
        k_values_det,
        progress_bar=True,
    )

    losses = np.asarray(jax.device_get(svi_result.losses))
    posterior_latent = {
        key: jnp.expand_dims(value, axis=0)
        for key, value in guide.median(svi_result.params).items()
    }
    predictive = infer.Predictive(
        model_psf_svi,
        posterior_samples=posterior_latent,
        return_sites=[
            "psf_det_model",
            "corr_field",
            "x_shift",
            "y_shift",
            "background",
            "flux_opt",
            "model_stack",
            "n_psf_corr",
            "rho_psf_corr",
            "sigma_psf_corr",
        ],
    )
    posterior = predictive(
        jax.random.PRNGKey(PSF_SVI_SEED + 1),
        sci_stack,
        err_stack,
        mask_stack,
        bkg_loc,
        bkg_scale,
        base_psf_det_jax,
        k_values_det,
    )

    psf_det_model = np.asarray(jax.device_get(posterior["psf_det_model"]))[0]
    corr_field = np.asarray(jax.device_get(posterior["corr_field"]))[0]
    flux_opt = np.asarray(jax.device_get(posterior["flux_opt"]))[0]
    background_post = np.asarray(jax.device_get(posterior["background"]))[0]
    model_stack = np.asarray(jax.device_get(posterior["model_stack"]))[0]
    x_shift = np.asarray(jax.device_get(posterior["x_shift"]))[0]
    y_shift = np.asarray(jax.device_get(posterior["y_shift"]))[0]
    residual = (np.asarray(image_stamps, dtype=float) - model_stack) / np.asarray(noise_stamps, dtype=float)

    header = fits.Header()
    header["BAND"] = band
    header["RUN_TAG"] = RUN_TAG
    header["NSTARS"] = len(image_stamps)
    header["BASEPID"] = int(base_pid)
    header["EXPTIME"] = float(exptime_sec)
    header["BGSIGMA"] = float(background_sigma_scalar)
    header["BGMED"] = float(background_level_scalar)
    header["SVIITER"] = int(PSF_SVI_MAX_ITER)
    header["SVISEED"] = int(PSF_SVI_SEED)
    header["LOSSFIN"] = float(losses[-1])

    model_psf_path = OUTPUT_DIR / f"{band}_psf_model_svi.fits"
    summary_path = OUTPUT_DIR / f"{band}_psf_svi_summary.json"

    save_fits(psf_det_model, header, model_psf_path, "PSFMODEL")

    psf_summary = {
        "band": band,
        "selected_psf_ids": [int(pid) for pid in resolved_psf_ids[band]],
        "base_psf_policy": BASE_PSF_POLICY,
        "base_psf_id": int(base_pid),
        "base_psf_center_pix": [float(base_center[0]), float(base_center[1])],
        "noise_model": "sigma_tot = sqrt(bg_sigma^2 + max(science-bg,0)/EXPTIME)",
        "exptime_sec": float(exptime_sec),
        "background_sigma_scalar": float(background_sigma_scalar),
        "background_level_scalar": float(background_level_scalar),
        "svi_iterations": int(PSF_SVI_MAX_ITER),
        "final_loss": float(losses[-1]),
        "psf_model_shape": list(np.asarray(psf_det_model).shape),
        "n_psf_corr": float(np.asarray(jax.device_get(posterior["n_psf_corr"]))[0, 0]),
        "rho_psf_corr": float(np.asarray(jax.device_get(posterior["rho_psf_corr"]))[0, 0]),
        "sigma_psf_corr": float(np.asarray(jax.device_get(posterior["sigma_psf_corr"]))[0, 0]),
        "model_psf_path": str(model_psf_path),
    }
    with open(summary_path, "w", encoding="utf-8") as fh:
        json.dump(psf_summary, fh, indent=2)
    psf_summaries[band] = psf_summary

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    axes[0].imshow(base_stamp_raw, origin="lower", cmap="twilight", norm=make_log_norm(base_stamp_raw))
    axes[0].set_title(f"{band.upper()} base star | id={int(base_pid)}")
    axes[1].imshow(base_psf_det, origin="lower", cmap="twilight", norm=make_log_norm(base_psf_det))
    axes[1].set_title("Initial base PSF")
    axes[2].imshow(psf_det_model, origin="lower", cmap="twilight", norm=make_log_norm(psf_det_model))
    axes[2].set_title("SVI PSF model")
    im_corr = axes[3].imshow(corr_field, origin="lower", cmap="coolwarm")
    axes[3].set_title("Correction field")
    axes[4].plot(losses)
    axes[4].set_yscale("asinh")
    axes[4].set_title("SVI loss")
    axes[4].set_xlabel("iteration")
    plt.colorbar(im_corr, ax=axes[3], fraction=0.046, pad=0.04)
    for ax in axes[:4]:
        ax.set_xticks([])
        ax.set_yticks([])
    fig.tight_layout()
    plt.show()

    n_show = len(image_stamps)
    fig, axes = plt.subplots(n_show, 3, figsize=(9, 2.6 * n_show), constrained_layout=True)
    axes = np.array(axes)
    if n_show == 1:
        axes = axes[None, :]

    for i in range(n_show):
        im0 = axes[i, 0].imshow(np.asarray(image_stamps[i]), origin="lower", cmap="twilight", norm=make_log_norm(image_stamps[i]))
        axes[i, 0].set_title(f"star {resolved_psf_ids[band][i]}")
        im1 = axes[i, 1].imshow(model_stack[i], origin="lower", cmap="twilight", norm=make_log_norm(model_stack[i]))
        axes[i, 1].set_title(f"model | dx={x_shift[i]:+.3f}, dy={y_shift[i]:+.3f}")
        im2 = axes[i, 2].imshow(residual[i], origin="lower", cmap="coolwarm", vmin=-3, vmax=3)
        axes[i, 2].set_title(f"residual | flux={flux_opt[i]:.3g}")
        for j in range(3):
            axes[i, j].set_xticks([])
            axes[i, j].set_yticks([])
    fig.colorbar(im0, ax=axes[:, 0].tolist(), fraction=0.02, pad=0.01)
    fig.colorbar(im1, ax=axes[:, 1].tolist(), fraction=0.02, pad=0.01)
    fig.colorbar(im2, ax=axes[:, 2].tolist(), fraction=0.02, pad=0.01)
    plt.show()

print(json.dumps(psf_summaries, indent=2))
print("Mask HTML:", ROOT / "Herculens" / "DSPL_LensModel" / "lens_arc_mask_drawer.html")

